<a href="https://colab.research.google.com/github/selim679/databricks-streaming-pipeline/blob/main/03_gold_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql.functions import *

df_silver = spark.read.format("delta").load("/Volumes/workspace/default/youtubeseries/delta/silver")
print(f"Silver table row count: {df_silver.count()}")
display(df_silver)

df_daily = df_silver.groupBy("event_date").agg(
    count("*").alias("total_events"),
    sum("amount").alias("total_revenue"),
    countDistinct("user_id").alias("active_users")
)

df_event_type = df_silver.groupBy("event_date", "event_type").agg(
    count("*").alias("event_count"),
    sum("amount").alias("revenue")
)

df_daily.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save("/Volumes/workspace/default/youtubeseries/delta/gold/daily_metrics")

df_event_type.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save("/Volumes/workspace/default/youtubeseries/delta/gold/event_metrics")

In [ ]:
display(spark.read.format("delta").load("/Volumes/workspace/default/youtubeseries/delta/gold/daily_metrics"))

In [ ]:
df_conversion = df_silver.groupBy("event_date").agg(
    (sum(when(col("event_type") == "purchase", 1).otherwise(0)) /
     sum(when(col("event_type") == "view", 1).otherwise(0))
    ).alias("conversion_rate")
)